# Scaling and Normalization

Imagine you have a dataset with two columns: **Age** (ranging from 18 to 80) and **Salary** (ranging from \$30,000 to \$150,000). 

Many machine learning algorithms (like K-Nearest Neighbors, Support Vector Machines, and Neural Networks) calculate the *mathematical distance* between data points. Because 150,000 is massively larger than 80, the algorithm will completely ignore the person's age and make all its decisions based solely on the salary. 

To fix this, we use **Feature Scaling** to force all our columns onto a level playing field without changing the underlying patterns in the data.

Let's set up a Python sandbox to see this in action using Scikit-Learn!

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

# Create a dataset with drastically different scales
data = {
    'customer_id': [1, 2, 3, 4, 5],
    'age': [25, 32, 45, 28, 55],               # Small numbers (20s to 50s)
    'salary': [50000, 65000, 120000, 55000, 90000] # Huge numbers (50k to 120k)
}

df = pd.DataFrame(data)

print("--- Original Data ---")
display(df)

--- Original Data ---


,customer_id,age,salary
0,1,25,50000
1,2,32,65000
2,3,45,120000
3,4,28,55000
4,5,55,90000


# 1. Min-Max Scaling (Normalization)
Min-Max Scaling shrinks all your data down so it fits perfectly between **0 and 1**. 
* The smallest number in the column becomes exactly `0.0`.
* The largest number in the column becomes exactly `1.0`.
* Everything else is squeezed proportionally in between.

**When to use it:** When you don't know the distribution of your data, or when an algorithm strictly requires bounded numbers (like Image Processing or some Neural Networks).

In [2]:
# 1. Initialize the Scaler
min_max_scaler = MinMaxScaler()

# 2. Create a copy of our data
df_minmax = df.copy()

# 3. Fit (learn the min/max) and Transform (apply the math)
df_minmax[['age', 'salary']] = min_max_scaler.fit_transform(df_minmax[['age', 'salary']])

print("--- Data after Min-Max Scaling ---")
display(df_minmax)

--- Data after Min-Max Scaling ---


,customer_id,age,salary
0,1,0.000000,0.000000
1,2,0.233333,0.214286
2,3,0.666667,1.000000
3,4,0.100000,0.071429
4,5,1.000000,0.571429


*(Notice how the youngest person is now Age `0.0`, and the poorest person is Salary `0.0`. The oldest person is Age `1.0`. Both columns are now playing by the exact same rules!)*

# 2. Standardization (Z-Score Scaling)
Standardization does not squeeze data into a strict 0-to-1 box. Instead, it centers the data around a **Mean of 0** and a **Standard Deviation of 1**. 

This means the average value becomes exactly `0.0`. Values above average become positive numbers (`1.5`), and values below average become negative numbers (`-0.8`).

**When to use it:** This is the default go-to for most Machine Learning models (like Linear Regression, Logistic Regression, and SVMs) because it handles outliers slightly better than Min-Max scaling and preserves normal distributions.

In [3]:
# 1. Initialize the Scaler
standard_scaler = StandardScaler()

# 2. Create a copy
df_standard = df.copy()

# 3. Fit and Transform
df_standard[['age', 'salary']] = standard_scaler.fit_transform(df_standard[['age', 'salary']])

print("--- Data after Standardization ---")
display(df_standard)

--- Data after Standardization ---


,customer_id,age,salary
0,1,-1.062321,-1.001483
1,2,-0.442634,-0.423704
2,3,0.708214,1.694817
3,4,-0.796741,-0.808890
4,5,1.593482,0.539260


*(Notice the negative numbers! A negative salary here doesn't mean the person is in debt; it just means they make less than the company average.)*

# 3. Robust Scaling (Handling Outliers)
What if Bill Gates walks into your dataset? 

If you use Min-Max scaling, Bill Gates' salary becomes `1.0`, and everyone else gets squeezed down to `0.000001`. The outlier ruins the scale. 
If you use Standardization, Bill Gates pulls the average so high that everyone else becomes heavily negative.

**RobustScaler** ignores the extreme highs and lows. It uses the Median and the Interquartile Range (IQR) to scale the data, making it "robust" (immune) to outliers.

In [4]:
# Let's add Bill Gates to our data
df_outlier = df.copy()
df_outlier.loc[5] = [6, 40, 10000000] # Age 40, Salary $10 Million

# Initialize Scalers
standard_scaler = StandardScaler()
robust_scaler = RobustScaler()

# Apply both to see the difference
df_standard_outlier = df_outlier.copy()
df_standard_outlier[['salary']] = standard_scaler.fit_transform(df_standard_outlier[['salary']])

df_robust_outlier = df_outlier.copy()
df_robust_outlier[['salary']] = robust_scaler.fit_transform(df_robust_outlier[['salary']])

print("--- Standardization WITH an Outlier ---")
print(df_standard_outlier['salary'].values) 
# Notice how the normal salaries are all squashed tightly around -0.4

print("\n--- Robust Scaling WITH an Outlier ---")
print(df_robust_outlier['salary'].values)
# Notice how the normal salaries remain reasonably spaced around 0.0, 
# while the outlier is pushed out to a massive 284.2!

--- Standardization WITH an Outlier ---
[-0.45423423 -0.45017857 -0.4353078  -0.45288234 -0.44341913  2.23602207]

--- Robust Scaling WITH an Outlier ---
[ -0.5         -0.22727273   0.77272727  -0.40909091   0.22727273
 180.40909091]


# 4. The Golden Rule of Scaling (`fit` vs `transform`)
When you eventually build a real predictive model, you will split your data into a **Training Set** (to teach the model) and a **Testing Set** (to test it). 

* **Rule 1:** You must scale BOTH sets.
* **Rule 2:** You must NEVER run `.fit_transform()` on the Testing Set. 

You should only `.fit()` the scaler on the Training data (learning the min/max or mean). Then, you `.transform()` the Test data using those saved rules. If you `fit` on the Test data, you are letting the model "peek" at the test answers, causing a critical error called **Data Leakage**. *(We will cover this deeply in Lesson 09!)*

---

## Real-World Use Case or Analogy:
Think of Scaling like **Comparing Student Grades from Different Countries**:

* **The Problem (Unscaled Data)**: You are a university admissions officer. Student A got a "90" on their math test in the USA. Student B got a "15" on their math test in France. If the algorithm just looks at the raw numbers, it thinks Student A is a genius and Student B failed miserably.
* **The Reality**: In the USA, tests are scored out of 100. In France, the top possible score is 20. Student B's score of 15 is actually fantastic!
* **Min-Max Scaling**: You force both grading systems onto a 0 to 1 scale. 
    * Student A: $90 / 100 = 0.90$
    * Student B: $15 / 20 = 0.75$
* **The Result**: By putting them on the exact same scale, the machine learning model can now clearly see that Student A performed slightly better, without being confused by the massive difference in the raw testing formats.

---